# Pass Probability
Using FMA to generate pitch control and calculate the probability will be completed.

---

**Change Directory**

This block of code is only ran so that, so this notebook can use the FMA library.

In [ ]:
from pathlib import Path
import sys
import os

cwd = os.getcwd()
os.chdir(cwd)

project_root = Path.cwd().parent
sys.path.append(str(project_root))

**Import FMA Library**

In [ ]:
from FootballMatchAnalysis.objects.match import Match

**Load Event and Tracking Data**

In [ ]:
DATADIR = './../data'
game_id = 1

In [ ]:
match = Match(DATADIR, game_id)

**Get a Moment**

We will start by getting a moment and reviewing all passing options

In [ ]:
# Get Moment
moment = match.get_moment(100)

# The include_player_velocities Parameters Will Allow Us to See What Direction Each Player is Moving
plot = moment.plot_moment(include_player_velocities=True)

**Overlay Pitch Control**

Let's now overlay a pitch control Voronoi diagram to see which parts of the pitch are controlled. The "red" an area of the pitch is, the more control the team on ball has over that part of the pitch, and the greater the likelihood a pass to that location will succeed. 

In [ ]:
plot = moment.plot_pitch_control(plot)

**Reviewing Passing Options**

Looking at this moment, Player 21 has a number of passing options. Given the way he's direction Player 21 is currently moving, he can probably play a pass to Player 17, 18, 20, 22. Let's now use the Moment method `pass_probability` to calculate the probability that a pass can be completed to these 4 players.

A note on these probability calculations: This is of course built on the great work by [Friends of Tracking](https://www.youtube.com/@friendsoftracking755). Moment's `pass_probability` acts simply as a wrapper to make pass probability calculations for any moment incredibly simple. These probabilities are calculated with consideration to number factors including, each player's distance from the target, each player's current velocity and the time it would take the ball to reach its target.

In [ ]:
# Players
players_to_consider = ["17", "18", "20", "22"]
away_team = moment.away_team()

# Iterate Over Players
for player in away_team:
    if player.name in players_to_consider:
        # Get Player's Location
        target = (player.x, player.y)
        
        # Calculate Probability of Pass to a Target
        probability = moment.pass_probability(target)

        print(f"A pass to Player{player.name} as a {round(100*probability, 2)}% chance of success")

**Calculating the Expected Pass Value**

xT allows us to assign a value to a pass to a given location. We can combine xT with pass probability to calculate the Expected Pass Value (EPV) for a given pass. This can then be used to help players evaluate their passing options.

In [ ]:
# Import xT Library
from FootballMatchAnalysis.analysis.xt import *

# Players
away_team = moment.away_team()

# Get Ball's Location
ball = moment.ball
ball_loc = (ball.x, ball.y)
ball_xt = get_xt(ball_loc, invert=True)
print(f"The xT of our current position is {ball_xt}")

# Get xT of Ball's Current Location

# Iterate Over Players
results = {}
for player in away_team:
    # Get Player's Location
    target = (player.x, player.y)
    
    # Calculate xT and Pass Probability
    target_xt = get_xt(target, invert=True)
    probability = moment.pass_probability(target)        
    print(f"  A pass to Player {player.name} will have an xT of {target_xt} but a probability of {round(100*probability, 2)}%")
    
    # Calcuate EPV (xT * Pass Probability)
    epv = target_xt * probability

    #Save Results
    results[player.name] = epv

# Find Best Passing Options
best_option = max(results, key=lambda player: results[player])
print(f"\n A pass to Player {best_option}, with an EPV of {round(results[best_option], 5)}, is our best passing option ")

# Plot Best Option
player23 = None
away_team = moment.away_team()
for player in away_team:
    if player.name == best_option:
        player23 = player

plot = moment.plot_moment()
plot = moment.plot_pitch_control(plot)
plot.draw_line(ball_loc, (player23.x, player23.y))